### Homework: Monitoring

In this homework, we will explore an alternative: OpenTelemetry (OTel). This is the industry standard for code instrumentation. Every monitoring framework we mentioned is built on top of it - like Logfire, Langfuse, Arize Phoenix and others.


In this homework we will use OTel directly. We will instrument our RAG with traces, capture metrics as span attributes, persist the spans to SQLite, and build a dashboard from the trace data.


We keep using the same course-lessons RAG from homework 1. The knowledge base is the 72 lesson pages pulled from GitHub, indexed with minsearch.

The starter loads the 72 course lessons, builds a text-search index, and wraps it in a RAGBase instance you can call right away:

#### Setup

Download starter code at:

PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/05-monitoring

* wget $PREFIX/rag_helper.py

  
* wget $PREFIX/starter.py

In [2]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

The loop keeps calling the model with `while True`, then checks whether the model returned any `function_call` items.

- If there are function calls, it runs the tool, appends the tool output to `messages`, and loops again.
- If there are no function calls, it `break`s.

So the stop condition is: **no function calls in the response**.


#### OpenTelemetry setup
First, install the OpenTelemetry libraries:
uv add opentelemetry-api opentelemetry-sdk

opentelemetry-api is the interface - the classes and functions you import in your code (trace, Tracer, Span)

opentelemetry-sdk is the implementation that actually creates and processes spans.

##### OpenTelemetry

Before we start, we need to learn a few concepts from OTel - we will use them in this homework.

* A trace is the end-to-end story of a single request as it moves through your system. For us, it's one RAG call.
* A span is one operation within a trace. A trace is made of one or more spans, organized as a tree. Each span has a name, a start and end time, and a set of attributes. For us we will have one span inside the trace, but for agents one trace will have multiple spans.
* Attributes are key-value pairs attached to a span - anything you want to record, like the number of tokens used or the cost of a call.

When a span finishes - meaning the code block it wraps completes - the SDK hands it to a span processor, which forwards it to an exporter. 

The exporter decides where the span goes: to the console, to a file, to a database, or to a remote collector. We will see all of this in practice in the questions below.

We start with the ConsoleSpanExporter, which prints each finished span to the terminal so we can see what OTel captures:

In [3]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

Here is what each line does:

* TracerProvider() creates the SDK's central configuration object. It owns the span processors and decides how spans are built.

* SimpleSpanProcessor(ConsoleSpanExporter()) wires a processor that forwards every finished span to the console exporter, one at a time. "Simple" means synchronous and immediate - good for development.

* trace.set_tracer_provider(provider) registers the provider globally, so every call to trace.get_tracer(...) returns a tracer backed by it.

* trace.get_tracer("llm-zoomcamp") returns a Tracer we use to create spans. The string is just a label for the instrumentation scope - it identifies which part of the code produced the spans.


Put this block at the top of your script, before you import or use starter - so the tracer provider is ready before any code that might create spans.

With the tracer in hand, you can wrap any block of code in a span:

```
with tracer.start_as_current_span("my_operation") as span:
    # your code here
    span.set_attribute("my_key", "my_value")
```

*start_as_current_span* creates a new span and makes it the "current" span for the duration of the with block. Any code inside the block - including other calls to *start_as_current_span* - becomes a child of this span. When the block exits, the span ends automatically.

You will use this pattern to instrument the RAG methods in the questions below.

### Q1. First trace

Wrap the *rag()* method so each call produces a span. The simplest way is to create a *RAGTraced* subclass of *RAGBase* that wraps *rag()*, *search()*, and *llm()* each in their own span.

Run this query:

How does the agentic loop keep calling the model until it stops?

The console exporter prints every finished span as a dictionary. Count the spans in the console output - each one is a separate ReadableSpan entry. How many spans does the trace produce?

* 1 
* 3 <--
* 5
* 7

In [7]:

with tracer.start_as_current_span("running_rag") as span:
    
    from starter import rag

    query = "How does the agentic loop keep calling the model until it stops?"
    answer = rag.rag(query)
    print(answer)

    span.set_attribute("level", "user_operation")
    #output: 1 span


It keeps calling the model in a `while True` loop.

Each iteration:
1. Send the full message history to the model.
2. Check the response for any `function_call` items.
3. If there are tool calls, run them and append the results to `messages`.
4. If there are no function calls, `break` out of the loop.

So the stop condition is: **no function calls in the model’s response**.
{
    "name": "running_rag",
    "context": {
        "trace_id": "0xca3160b1580567c3419cb72acdab1d5a",
        "span_id": "0xb4a3dce3e4d80b0f",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-20T02:59:45.213674Z",
    "end_time": "2026-07-20T02:59:47.524280Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "level": "user_operation"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",


#### another way to do Q1

In [2]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

with tracer.start_as_current_span("running_rag") as span:
    
    from starter import index, client
    from rag_helper import RAGTraced
    
    rag = RAGTraced(index=index, llm_client=client)
    answer = rag.rag("How does the agentic loop keep calling the model until it stops?")
    span.set_attribute("level", "user_operation")
 #output: 3 span

{
    "name": "search_def",
    "context": {
        "trace_id": "0x37a8c59b8afc8296fbcaf68876e29ef2",
        "span_id": "0x490cb9b5d9c09709",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xd5efe2c6eb397514",
    "start_time": "2026-07-20T03:11:40.599446Z",
    "end_time": "2026-07-20T03:11:40.601181Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "level": "def_operation"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "49a6d8c7-9c58-47a1-b6f1-e6d801099830",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm_def",
    "context": {
        "trace_id": "0x37a8c59b8afc8296fbcaf68876e29ef2",
        "span_id": "0x28fc3c0f40da54b3",
        "t

### Q2. Capturing metrics as span attributes

Spans are not just timing markers - you can attach any information you want to them with set_attribute. We already use spans to record how long each step takes. Now we'll add the metrics we care about: tokens and cost.

Read the token usage from the LLM response (the llm() method in the starter already returns the raw response object) and set them as attributes on the llm span:

```
span.set_attribute("input_tokens", usage.input_tokens)
span.set_attribute("output_tokens", usage.output_tokens)
```

And since we know both input and output tokens, we can also compute the cost using the code from the previous modules.

Now re-run the query. How many input tokens do we see?

* 700
* 7000 <--
* 70000
* 700000

In [3]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

with tracer.start_as_current_span("running_rag") as span:
    
    from starter import index, client
    from rag_helper import RAGTraced
    
    rag = RAGTraced(index=index, llm_client=client)
    answer = rag.rag("How does the agentic loop keep calling the model until it stops?")
    span.set_attribute("level", "user_operation")

#output: 7111

Overriding of current TracerProvider is not allowed


{
    "name": "search_def",
    "context": {
        "trace_id": "0x0b12ca198d17c8bb3e144d29ef6c4def",
        "span_id": "0xed684ceb8009d8d0",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x143d8e137d1d16a3",
    "start_time": "2026-07-20T03:13:01.904549Z",
    "end_time": "2026-07-20T03:13:01.908349Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "level": "def_operation"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "49a6d8c7-9c58-47a1-b6f1-e6d801099830",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm_def",
    "context": {
        "trace_id": "0x0b12ca198d17c8bb3e144d29ef6c4def",
        "span_id": "0xc5ad99bf717a1e4d",
        "t

### Q3. Span timing

Each span automatically records its duration. 

Look at the console output from Q1 and find the durations for the search span and the llm span.

For a typical query, roughly how long does the LLM call take?

* Under 100ms
* 100-500ms
* 500-2000ms
* Over 2000ms <--

span times:

* search_def: 3.8 ms (908349 − 904549 µs)
* llm_def: 2000.265 ms (≈2.0 s)

### Q4. Saving traces to SQLite

Right now the spans are printed to the terminal and then gone. We don't save them.

We want to persist them so we can query them later.

In this homework, we'll use SQLite - it's a more lightweight option than Postgres, so we don't need to set up any docker containers in this homework.

Our instrumentation is already done, we don't need to change anything there. But we need to create a custom exporter. Instead of printing the spans, it will save them to the database.

OTel calls the exporter through the same span processor we already use, we just swap the destination.

Now we will create a custom exporter that saves each finished span to a SQLite database. The exporter extends SpanExporter. It has the following methods:

* export method that receives a list of ReadableSpan objects
* shutdown and force_flush methods

In [2]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

Replace the console exporter with this new exporter:

```
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)
```

In [2]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

with tracer.start_as_current_span("running_rag") as span:
    
    from starter import index, client
    from rag_helper import RAGTraced
    
    rag = RAGTraced(index=index, llm_client=client)
    answer = rag.rag("How does the agentic loop keep calling the model until it stops?")
    span.set_attribute("level", "user_operation")

Re-run the query from Q1. Which span names appear in the spans table?

* Only rag
* rag and llm
* rag, search, and llm <--
* search, llm, and judge

In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("traces.db")
df = pd.read_sql_query("SELECT DISTINCT name FROM spans;", conn)
print(df)
conn.close()

          name
0   search_def
1      llm_def
2      rag_def
3  running_rag


### Q5. Querying trace data

The traces are now in SQLite. Run one more query through the traced RAG, then query the database.

The rag span wraps everything, so its duration includes both search and llm. To see where time actually goes, exclude the rag span and compare the children.

Using SQL (or pandas), compute the total duration for each span name excluding rag. Which span type takes the most total time?

* search
* llm <--
* They're all about the same

In [3]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

with tracer.start_as_current_span("running_rag") as span:
    
    from starter import index, client
    from rag_helper import RAGTraced
    
    rag = RAGTraced(index=index, llm_client=client)
    answer = rag.rag("Can I still join the course?")
    span.set_attribute("level", "user_operation")

In [8]:
with tracer.start_as_current_span("running_rag") as span:
    
    from starter import index, client
    from rag_helper import RAGTraced
    
    rag = RAGTraced(index=index, llm_client=client)
    answer = rag.rag("Can I still join the course?")
    span.set_attribute("level", "user_operation")

Using SQL (or pandas), compute the total duration for each span name excluding rag. Which span type takes the most total time?

In [11]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("traces.db")
df = pd.read_sql_query("""
    SELECT name, start_time, end_time
    FROM spans
    WHERE name NOT IN ('rag_def', 'running_rag')
""", conn)
conn.close()

df["duration_ms"] = (df["end_time"] - df["start_time"]) / 1_000_000

totals = df.groupby("name")["duration_ms"].sum().sort_values(ascending=False)
print(totals)


name
llm_def       7884.7361
search_def       9.0525
Name: duration_ms, dtype: float64


### Q6. Token stability across runs

Load the SQLite data with pandas. One thing a dashboard can tell you is how stable your system is. If the same query always produces the same number of input tokens, the context your RAG retrieves is consistent. If it varies a lot, something in the search may be unstable.

Run the same query from Q1 three more times (so you have 4 RAG calls total in the database). Then compute the input tokens for each llm span.

How much do the input tokens vary across these 4 runs?

* They're identical
* Within 10% of each other
* Within 50% of each other
* They vary more than 50%

In [14]:
with tracer.start_as_current_span("running_rag") as span:
    
    from starter import index, client
    from rag_helper import RAGTraced
    
    rag = RAGTraced(index=index, llm_client=client)
    answer = rag.rag("How does the agentic loop keep calling the model until it stops?")
    span.set_attribute("level", "user_operation")

In [16]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("traces.db")
df = pd.read_sql_query("""
    SELECT *
    FROM spans
    WHERE name NOT IN ('rag_def', 'running_rag', 'search_def')
""", conn)
conn.close()

df["duration_ms"] = (df["end_time"] - df["start_time"]) / 1_000_000

print(df)

      name           start_time             end_time  input_tokens  \
0  llm_def  1784517944073045000  1784517946672780300          7111   
1  llm_def  1784518438945338000  1784518440151708900          5676   
2  llm_def  1784518484135642100  1784518486453366000          5676   
3  llm_def  1784519192653344300  1784519194414250300          5676   
4  llm_def  1784519754450499800  1784519756448184200          7111   
5  llm_def  1784519760914711000  1784519763231938000          7111   
6  llm_def  1784519765265293100  1784519766957775200          7111   

   output_tokens  cost  duration_ms  
0             98  None    2599.7353  
1             31  None    1206.3709  
2             31  None    2317.7239  
3             28  None    1760.9060  
4            104  None    1997.6844  
5             93  None    2317.2270  
6            109  None    1692.4821  


How much do the input tokens vary across these 4 runs?

* They're identical <--
* Within 10% of each other
* Within 50% of each other
* They vary more than 50%